# Module 12 Lab - Ethics, Fairness, and Bias in ML

By

- Viktoriya Kurmisheva
- Tien Manh Nguyen
- Zaid Alayedy


**Objective:**
To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.

In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.

## Part 1: What is Algorithmic Bias?

**Concept:**
Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.

**Sources of Bias:**

Historical Bias: The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).
Measurement Bias: The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).
Representation Bias: The data underrepresents certain groups, so the model doesn't learn to perform well for them.

**Problem:**
We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like sex and race, which we can use to audit our model for bias.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

df = pd.read_csv(
    url,
    header=None,
    names=columns,
    sep=',\s*',
    engine='python',
    na_values='?'
)

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

# Features and target
X = df.drop('income', axis=1)
y = df['income']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model
model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

# Evaluate
print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

<>:23: SyntaxWarning: invalid escape sequence '\s'
<>:23: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_535/2379571722.py:23: SyntaxWarning: invalid escape sequence '\s'
  sep=',\s*',


Overall model accuracy: 84.61%


## Part 2: Auditing the Model for Fairness

High overall accuracy can hide poor performance on specific subgroups. To properly evaluate the model, we need to audit its performance across sensitive attributes, such as sex.

### Concept: Group Fairness

One common fairness goal is to ensure that the model performs equally well for different groups.

We can evaluate this by:

Calculating performance metrics (like accuracy)
Separately for each subgroup (e.g., males vs females)
### Your Task
Create a function that:
Filters the dataset by a subgroup (e.g., sex = Male)
Computes the model’s accuracy on that subgroup
Use the function to calculate:
Accuracy for Males
Accuracy for Females
Compare the results:
Does one group have higher accuracy?
Is the difference significant?

In [ ]:
# --- ENTER YOUR CODE HERE ---

def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""

    # 1. Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)


# 4. Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%



## Task 2: Deeper Dive with a Confusion Matrix

Accuracy alone doesn't tell the whole story. To better understand model behavior, we analyze the types of errors it makes for each group.

### Your Task

Calculate and compare the False Positive Rate (FPR) and False Negative Rate (FNR) for males and females.

### Definitions

**False Positive Rate (FPR):**

FPR =FP /(FP+TN)


The percentage of people who did NOT have high income but were incorrectly predicted to have high income.

**False Negative Rate (FNR):**

FNR= FN /(FN+TP)

The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [ ]:
from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    # 1. Filter subgroup
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 2. Predictions
    y_pred_subgroup = model.predict(X_subgroup)

    # 3. Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    # 4. Rates
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    return fpr, fnr


# --- ENTER YOUR CODE HERE ---

# 1. Calculate the rates for males and females
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

# 2. Print results
print(f"Male - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%



## Reflective Knowledge Check

### 1. Analyze Your Results

**From the subgroup accuracy results:**

**Accuracy for Males:** 81.20%  
**Accuracy for Females:** 91.81%

The results show a noticeable difference in accuracy between males and females. The model performs better for females, with an accuracy of 91.81%, compared to 81.20% for males. This is a difference of 10.61 percentage points. This suggests that the model may favor females in terms of overall prediction accuracy. However, we would need further testing to know whether this difference is statistically significant. These results show that high overall accuracy does not necessarily mean the model performs equally well for everyone.

### 2. Interpret the Errors

**From the calculated rates:**

**Male FPR:** 10.26%  
**Male FNR:** 37.80%  
**Female FPR:** 2.81%  
**Female FNR:** 47.84%

The model is more likely to make a False Positive error for males because their FPR is 10.26%, compared to 2.81% for females. This means it incorrectly predicts high income for low-income males more often. On the other hand, females have a higher False Negative Rate, meaning the model is more likely to incorrectly classify high-income females as low-income.

**Real-world consequence (loan example):**

If a bank used this model to help approve loans, a False Positive could lead to approving someone who does not actually have the income the model predicted. This could increase the risk of missed payments or financial losses. Since males have a higher FPR, they might receive this type of incorrect approval more often. At the same time, the higher FNR for females could create disadvantages if qualified applicants were incorrectly rejected. This shows how different prediction errors can affect groups in different ways.

### 3. Justify a Decision (Hiring Scenario)

**In a hiring context:**

A False Positive could mean selecting a candidate who does not meet the expected qualifications, while a False Negative could mean rejecting someone who actually has the potential to succeed.

**More harmful error:**

I believe a False Negative is generally more harmful in this situation because it could prevent a qualified person from getting a job. Someone might have the necessary skills and experience but lose the opportunity because the model makes an incorrect prediction.

**Based on the results:**

Females have a higher FNR of 47.84%, compared to 37.80% for males. If these predictions were used for hiring, this difference could create an unfair disadvantage for female applicants. Meanwhile, the higher FPR for males could lead to more incorrect positive decisions for that group.

**Decision:**

I would not approve this model for deployment in a hiring system.

**Justification:**

The differences in error rates raise concerns about fairness because the model does not make the same types of mistakes equally across both groups. More importantly, predicting someone's income is not the same as evaluating their qualifications for a job. Using this model for hiring could lead to unfair decisions and missed opportunities for qualified applicants. I think the model needs more evaluation and improvements before it could be considered for such an important purpose.

### 4. Propose a Mitigation

Removing the 'sex' column would not completely eliminate bias because other features could still contain information related to gender.

**Why?**

Features such as occupation, education, hours-per-week, and marital-status may have patterns that are connected to gender. Even without the 'sex' column, the model could still learn these relationships and produce different outcomes for males and females.

**This is called:**

Proxy bias, which happens when other features indirectly reveal information about a sensitive attribute.

**Conclusion:**

I think removing sensitive information is a useful step, but it does not guarantee fairness. The model might still make unfair predictions based on patterns hidden in other features.

**True fairness requires:**

We should examine the features more carefully, consider techniques such as reweighting or fairness constraints, and test the model again after making changes. Comparing accuracy, FPR, and FNR for both groups would help us understand whether the changes actually reduced bias.
